# 02. Очистка данных по шагам
Блокнот вызывает общие функции из `src/cleaning.py` по одной и показывает результат каждого шага. Тот же код запускается из командной строки: `python -m src.pipeline`. Исполняется целиком после перезапуска ядра.

In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))
import logging
import pandas as pd
from IPython.display import display, Markdown
from src import config as cfg, io_utils as io, cleaning as cl, validation
logging.basicConfig(level=logging.WARNING)
pd.set_option('display.max_columns', 60); pd.set_option('display.width', 220); pd.set_option('display.max_colwidth', 80)
MODE = 'debug'          # 'debug' - срез 200 тыс. строк, 'full' - весь набор
sfx = '_debug' if MODE == 'debug' else ''
SRC = cfg.DEBUG_SLICE if MODE == 'debug' else cfg.RAW_BRANDED
OUT_DIR = cfg.RESULTS / ('debug_nb' if MODE == 'debug' else 'full_nb')
stats = cl.Stats()

## Исходная таблица с явной схемой типов
Чтение с параметром `dtype` из `config.SCHEMA`; `keep_default_na=False` — заглушки вроде `n/a` не исчезают молча.

In [ ]:
if MODE == 'debug' and not SRC.exists():
    io.make_slice()
df = io.read_csv_typed(SRC)
n0, c0 = df.shape
print(n0, c0, round(df.memory_usage(deep=True).sum() / 1e6, 1), 'МБ')
display(df.dtypes.astype(str).value_counts())

## Шаг 1. Соединение с `food.csv` по `fdc_id`
Левое соединение 1:1 (ключ справочника проверен на уникальность). Контроль: число строк до и после и число строк без пары.

In [ ]:
df = cl.join_food(df, stats)
display(stats.frame().tail(1))

## Шаг 2. Имена столбцов и типы

In [ ]:
df = cl.normalize_column_names(df, stats)
df = cl.cast_types(df, stats)
display(stats.frame().tail(2))

## Шаг 3. Скрытые пропуски
`n/a`, `none`, `nan`, пустые после обрезки пробелов, порция ≤ 0, даты до 1990 г. → `NA`.

In [ ]:
before = df.isna().sum()
df = cl.replace_hidden_missing(df, stats)
display((df.isna().sum() - before).loc[lambda s: s > 0].to_frame('добавлено NA'))

## Шаг 4. Структура пропусков
Пропуски не удаляются и не заполняются модой или медианой: это выдумало бы данные. Проверяем, не сосредоточены ли они в одной группе (`data_source`).

In [ ]:
df = cl.analyze_missing(df, stats, OUT_DIR)
display(pd.read_csv(OUT_DIR / 'missing_by_data_source.csv', index_col=0))

## Шаг 5. GTIN и повторы
Коды дополняются нулями до 14 знаков; полные дубликаты и повторы `fdc_id` удаляются; прежние версии одного GTIN помечаются флагом `is_latest_version`.

In [ ]:
n = len(df)
df = cl.normalize_gtin(df, stats)
df = cl.drop_duplicates(df, stats)
print('строк было', n, 'стало', len(df))
display(df['is_latest_version'].value_counts())

## Шаг 6. Категории
Пробелы, регистр, словарь единиц (`GRM/GM→g`, `MLT→ml`, `MC→mcg`), редкие категории → группа `Other`.

In [ ]:
df = cl.normalize_categories(df, stats)
display(df['serving_size_unit'].value_counts(dropna=False))
display(df['branded_food_category_grp'].value_counts().head(8))

## Шаг 7. Каналы продаж и текст
`trade_channel` — JSON-список (нарушение 1НФ) → `A|B` без повторов; из `ingredients` удаляется HTML, нормализуются пробелы.

In [ ]:
df = cl.parse_trade_channel(df, stats)
df = cl.clean_text(df, stats)
display(df['trade_channel'].dropna().value_counts().head(5))

## Шаг 8. Флаги пропусков и единая шкала
`serving_size_std` — порция в граммах или миллилитрах; `package_weight` разбирается в `package_weight_g` / `package_weight_ml`.

In [ ]:
df = cl.add_missing_flags(df, stats)
df = cl.normalize_units(df, stats)
display(df[['serving_size', 'serving_size_unit', 'serving_size_std', 'serving_unit_std',
            'package_weight', 'package_weight_g', 'package_weight_ml']].dropna(subset=['package_weight']).head(6))

## Шаг 9. Предметные ограничения и выбросы
Даты не в будущем; `discontinued_date` не раньше `available_date`. Выбросы порции — двумя способами (IQR и MAD), без удаления: ставятся флаги.

In [ ]:
df = cl.check_domain(df, stats)
df = cl.flag_outliers(df, stats, OUT_DIR)
import json; print(json.dumps(json.load(open(OUT_DIR / 'outliers.json')), indent=1)[:900])

## Итог: «было — стало»

In [ ]:
df = cl.finalize(df, stats)
display(stats.frame())
print(f'Было {n0} x {c0}, стало {len(df)} x {df.shape[1]}; потеряно строк: {100 * (n0 - len(df)) / n0:.4f} %')

## Сохранение и сверка с командной строкой
Результат блокнота пишется в отдельный файл и сравнивается с результатом `python -m src.pipeline` построчно.

In [ ]:
out = cfg.PROCESSED / f'clean_notebook{sfx}.parquet'
io.write_parquet(df, out)
cli = cfg.PROCESSED / f'clean{sfx}.parquet'
if cli.exists():
    validation.compare(cli, out)
else:
    print('Сначала выполните: python -m src.pipeline', '--mode debug' if MODE == 'debug' else '')

## Идемпотентность
Повторная очистка готовой таблицы ничего не меняет.

In [ ]:
again = cl.clean_all(df.copy(), cl.Stats(), None)
pd.testing.assert_frame_equal(df, again, check_exact=True)
print('идемпотентность подтверждена')